# Engine: a chess engine at a chosen strength

**Deliverable:** "a chess engine that plays at a chosen strength" (README, Mark-I scope) and the
analysis that the teaching side (`gambit_teach`) uses to grade the human's moves.

This notebook wraps any UCI engine (Stockfish by default; check its GPL license against the
project's license plan, see README "License") through `python-chess`, plus two tiny pure-Python
engines (`RandomEngine`, `MaterialEngine`) so every other notebook can be tested without a binary.

All engines share one small interface (`EngineLike`):

* `play(board) -> chess.Move` : the move the robot will make.
* `analyse(board, multipv=1, root_moves=None) -> list[Candidate]` : best lines, best first, each
  scored in centipawns **from the point of view of the side to move**.
* `close()`.

**TODO checklist**
- [ ] `strength_options` (pure, test it first), `score_to_cp`.
- [ ] `Engine` wrapper: lifecycle (lazy open/close/context manager), `play`, `analyse`.
- [ ] Fallback engines: `RandomEngine`, `MaterialEngine` (negamax + alpha-beta), `make_engine`.
- [ ] Self-checks (plan at the bottom).
- [ ] Install Stockfish (`sudo apt install stockfish` or download) and set `engine.path` in
      `gambit_bringup/config/game.yaml`.
- [ ] Decide how to play below Stockfish's minimum `UCI_Elo` (1320): Skill Level mapping, or the
      teacher policy's own move choice (see `strength_options`).
- [ ] Calibrate the Elo settings against real targets in `run_engine_strength.ipynb` (~1700).
- [ ] Use two engine instances: a strength-limited *player* and a full-strength *analyst*
      (limiting strength also weakens the analysis Stockfish reports).

In [ ]:
from __future__ import annotations

import random
import shutil
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Protocol, Sequence

import chess
import chess.engine
import yaml

## Configuration

Every setting lives in `gambit_bringup/config/game.yaml`. The path is relative to the notebook's
folder, which works both from `gambit_game/notebooks` and `gambit_teach/notebooks`.

(`load_game_config` is left implemented: it is two lines of config I/O and every notebook in
gambit_game and gambit_teach needs `GAME_CFG` at import time.)

In [ ]:
GAME_CFG_PATH = Path("../../gambit_bringup/config/game.yaml")


def load_game_config(path: Path | str = GAME_CFG_PATH) -> dict:
    """Load game.yaml as a dict."""
    with open(path) as f:
        return yaml.safe_load(f)


GAME_CFG = load_game_config()
ENGINE_CFG = GAME_CFG["engine"]
MATE_SCORE_CP = GAME_CFG["teaching"]["mate_score_cp"]

## Shared types

A `Candidate` is one engine line. `score_cp` is always from the side to move's point of view
(positive = good for the player about to move). A forced mate is turned into a large centipawn
value (`mate_score_cp` minus the distance to mate) so that every score is a plain number.

In [ ]:
@dataclass
class Candidate:
    move: chess.Move
    score_cp: int                       # side-to-move POV
    pv: list = field(default_factory=list)
    mate: Optional[int] = None          # moves to mate (+ we mate, - we get mated), if any

    def __repr__(self) -> str:
        """Short form for debugging, e.g. Candidate(e2e4, +35cp) or Candidate(a1a8, +9999cp mate 1)."""
        # Design: UCI move, signed centipawns, mate distance only when present.
        # TODO: implement
        raise NotImplementedError


class EngineLike(Protocol):
    def play(self, board: chess.Board) -> chess.Move: ...
    def analyse(self, board: chess.Board, multipv: int = 1,
                root_moves: Optional[Sequence[chess.Move]] = None) -> list[Candidate]: ...
    def close(self) -> None: ...


def score_to_cp(score: chess.engine.PovScore, pov: chess.Color,
                mate_score: int = MATE_SCORE_CP) -> tuple[int, Optional[int]]:
    """python-chess PovScore -> (centipawns from `pov`, mate distance or None)."""
    # Design:
    # - Turn the PovScore to the requested point of view first (PovScore.pov), then read the
    #   centipawns with a mate_score so a forced mate becomes a large finite number, and read the
    #   mate distance separately (Score.mate()).
    # TODO: implement
    raise NotImplementedError

## Strength limiting

Stockfish offers two knobs:

* `UCI_LimitStrength=true` + `UCI_Elo=<elo>`: targets a rating (Stockfish accepts roughly 1320..3190;
  the exact range depends on the version, read `engine.options["UCI_Elo"]`).
* `Skill Level` 0..20: older, coarser, not calibrated to Elo.

`strength_options` turns the config into the UCI option dict. It is pure, so it is easy to test.

In [ ]:
def strength_options(elo: Optional[int] = None, skill: Optional[int] = None,
                     cfg: dict = ENGINE_CFG) -> dict:
    """UCI options for the requested strength. `skill` wins over `elo`; neither = full strength.

    Returns a dict of UCI option names -> values, ready for engine.configure.
    """
    # Design:
    # - skill given -> {"Skill Level": skill}; validate the 0..20 range (ValueError otherwise).
    # - neither given -> explicitly turn strength limiting off (UCI_LimitStrength False).
    # - elo given -> UCI_LimitStrength True + UCI_Elo, kept inside cfg["min_elo"]..cfg["max_elo"].
    # Decisions for you: what to do below min_elo (clamp for now? map to Skill Level? let the
    # teacher pick weaker multipv moves?). Whatever you pick, make it visible to the caller.
    # TODO: implement
    raise NotImplementedError

## UCI engine wrapper

`Engine` starts the binary lazily, so constructing it is free and errors appear at the first use.
`find_engine_binary` returns `None` if the binary is not installed (the self-check then skips).

In [ ]:
def find_engine_binary(path: Optional[str] = None) -> Optional[str]:
    """Resolve the configured engine path (default ENGINE_CFG["path"]); None if not installed."""
    # Design: accept either a real file path or a command name on PATH (shutil.which).
    # TODO: implement
    raise NotImplementedError


class Engine:
    """python-chess UCI engine at a chosen strength.

    Use as a context manager (`with Engine(elo=1700) as eng:`) or call close().
    """

    def __init__(self, path: Optional[str] = None, elo: Optional[int] = None,
                 skill: Optional[int] = None, cfg: dict = ENGINE_CFG):
        """Store settings only; the process is started lazily in open()."""
        self.cfg = cfg
        self.path = path or cfg["path"]
        self.elo = elo
        self.skill = skill if skill is not None else cfg.get("skill_level")
        self._engine: Optional[chess.engine.SimpleEngine] = None

    # -- lifecycle --------------------------------------------------------------
    def open(self) -> "Engine":
        """Start the UCI process if not running; returns self."""
        # Design:
        # - find_engine_binary; a clear FileNotFoundError if missing (point at game.yaml).
        # - chess.engine.SimpleEngine.popen_uci, then configure Threads / Hash from cfg plus
        #   strength_options. Only send options the engine actually reports (engine.options),
        #   so other UCI engines do not choke on Stockfish-specific names.
        # TODO: implement
        raise NotImplementedError

    def close(self) -> None:
        """Quit the process (idempotent)."""
        # TODO: implement
        raise NotImplementedError

    def __enter__(self):
        # TODO: open and return self
        raise NotImplementedError

    def __exit__(self, *exc):
        # TODO: close
        raise NotImplementedError

    def set_strength(self, elo: Optional[int] = None, skill: Optional[int] = None) -> None:
        """Change strength; reconfigure the running process if there is one."""
        # Design: store the new values; if the engine is open, configure it with strength_options
        # (filtered to supported options, as in open()).
        # TODO: implement
        raise NotImplementedError

    # -- EngineLike ---------------------------------------------------------------
    def play(self, board: chess.Board, time_s: Optional[float] = None) -> chess.Move:
        """The engine's move under a time limit (default cfg["think_time_s"])."""
        # Design: open lazily, chess.engine.Limit(time=...), SimpleEngine.play(...).move
        # TODO: implement
        raise NotImplementedError

    def analyse(self, board: chess.Board, multipv: int = 1,
                root_moves: Optional[Sequence[chess.Move]] = None,
                time_s: Optional[float] = None, depth: Optional[int] = None) -> list[Candidate]:
        """Top `multipv` lines as Candidates, best first, side-to-move POV.

        `root_moves` restricts the search (used to score the move a human actually played at the
        same depth as the best move).
        """
        # Design:
        # - Limit by depth if given (or cfg["analysis_depth"]), else by time (cfg["analysis_time_s"]).
        #   Fixed depth makes CPL comparable across positions; fixed time is cheaper to reason about.
        # - SimpleEngine.analyse(..., multipv=..., root_moves=...) returns a dict for multipv=1 and
        #   a list otherwise: normalise to a list.
        # - Skip infos without a PV; convert scores with score_to_cp(score, board.turn); sort.
        # TODO: implement
        raise NotImplementedError

## Fallback engines (no binary needed)

* `RandomEngine`: uniformly random legal move; all analysis scores are 0. Useful as a "human".
* `MaterialEngine`: negamax with alpha-beta to a fixed depth on material only (P=100, N=B=300,
  R=500, Q=900). Very weak, but it should take free pieces and see mate-in-one, which is enough to
  exercise the game loop and the teaching code end to end.

In [ ]:
PIECE_CP = {chess.PAWN: 100, chess.KNIGHT: 300, chess.BISHOP: 300,
            chess.ROOK: 500, chess.QUEEN: 900, chess.KING: 0}


def material_cp(board: chess.Board) -> int:
    """Material balance in centipawns from the side to move's point of view."""
    # Design: sum PIECE_CP over board.piece_map(), + for the side to move, - for the opponent.
    # TODO: implement
    raise NotImplementedError


class RandomEngine:
    def __init__(self, seed: Optional[int] = None):
        """Own random.Random(seed) so games are reproducible."""
        self.rng = random.Random(seed)

    def play(self, board: chess.Board) -> chess.Move:
        """A uniformly random legal move."""
        # TODO: implement
        raise NotImplementedError

    def analyse(self, board, multipv=1, root_moves=None):
        """Up to `multipv` random moves (restricted to root_moves if given), all scored 0."""
        # TODO: implement
        raise NotImplementedError

    def close(self):
        pass                                    # nothing to release


class MaterialEngine:
    def __init__(self, depth: int = 2, seed: Optional[int] = 0):
        """Search depth in plies; a seeded RNG for tie-breaks between equal moves."""
        self.depth = depth
        self.rng = random.Random(seed)

    def _negamax(self, board: chess.Board, depth: int, alpha: int, beta: int, ply: int) -> int:
        """Negamax value of `board` for the side to move, alpha-beta pruned."""
        # Design:
        # - Terminal nodes: checkmate = -(MATE_SCORE_CP - ply) (prefer faster mates, delay being
        #   mated); stalemate / insufficient material = 0. Depth 0 -> material_cp.
        # - Recurse with the negamax sign flip and swapped/negated window; cut when alpha >= beta.
        # - Move ordering matters for pruning: try captures first.
        # TODO: implement
        raise NotImplementedError

    def analyse(self, board, multipv=1, root_moves=None):
        """Score every root move (or root_moves) and return the best `multipv` Candidates."""
        # Design:
        # - Work on a copy of the board. Shuffle root moves with the seeded RNG (random tie-break).
        # - Search each root move with a full window, so every multipv score is exact (an aspiration
        #   window would only bound the non-best ones, which breaks CPL).
        # - Convert scores near MATE_SCORE_CP back into a mate distance in moves (+ / -).
        # TODO: implement
        raise NotImplementedError

    def play(self, board):
        """The best move from analyse."""
        # TODO: implement
        raise NotImplementedError

    def close(self):
        pass                                    # nothing to release


def make_engine(prefer_uci: bool = True, elo: Optional[int] = None) -> EngineLike:
    """Stockfish if installed (and preferred), else MaterialEngine."""
    # TODO: implement
    raise NotImplementedError

## Self-check

In [ ]:
# Self-check plan (write these as asserts once the functions above work):
# - strength_options: 1700 -> limit on with UCI_Elo 1700; 800 -> your below-min_elo decision;
#   skill=5 -> {"Skill Level": 5}; no args -> limit off; skill=25 -> ValueError.
# - MaterialEngine takes a hanging queen: FEN "4k3/8/8/3q4/8/8/8/3QK3 w - - 0 1" -> d1d5.
# - Mate in one (back rank): FEN "6k1/5ppp/8/8/8/8/8/R5K1 w - - 0 1", multipv=3 -> best is a1a8
#   with mate == 1, three candidates, sorted by score.
# - root_moves=[g1f1] returns exactly that one move.
# - RandomEngine(0).play(start) is a legal move.
# - If find_engine_binary() finds Stockfish: with Engine(elo=1700) analyse the start position with
#   multipv=3 and a short time; 3 legal candidates. Otherwise print that the check was skipped.
# Example shape of a test:
# b = chess.Board("4k3/8/8/3q4/8/8/8/3QK3 w - - 0 1"); assert MaterialEngine().play(b) == chess.Move.from_uci("d1d5")
print("Self-check: not written yet (see plan above)")